# Análisis de la morfología interna del fruto en Cranberry

En este tutorial, demostraremos cómo realizar el análisis de la morfología interna de frutos utilizando `FruitInternalAnalyzer`, una herramienta para extraer medidas de morfología y color a partir de una imagenes de rodajas transversales de cranberry.

El primer paso es cargar la clase `FruitInternalAnalyzer` desde Traitly para crear el objeto `cranberry`, que contendrá todo lo necesario para el análisis. Puedes elegir el nombre de objeto que prefieras.

A continuación, inicializamos la clase indicando la ubicación de nuestra imagen mediante el parámetro `path`.  


In [ ]:
from traitly.fruit_phenotyping import FruitInternalAnalyzer

In [ ]:
image_path = "./cranberry_slices.jpg"
cranberry = FruitInternalAnalyzer(path = image_path)

Primero, cargamos la imagen en el objeto mediante `load_image()`. Por defecto, la imagen se mostrará en pantalla (`plot=True`). Una vez cargada, puede accederse a ella a través de `cranberry.img`. Para más detalles sobre los datos almacenados en el objeto, consulta el apartado de [atributos de la clase].

In [ ]:
cranberry.load_image() 

A continuación, ejecutamos `setup_measurements()` para establecer el diámetro de las referencias de tamaño (círculos negros) y, opcionalmente, extraer la información del código QR de la etiqueta. Activando `plot = True` podemos inspeccionar en detalle la detección de la referencia y el diámetro en píxeles de cada círculo.

<br>

Como indican los resultados, se detectó una tira de círculos (`Ref 1`) compuesta por 6 círculos. Antes de calcular el promedio, `setup_measurements()` elimina los círculos cuya desviación estándar es mayor a 2, para evitar ruido por círculos mal detectados o muy diferentes al resto. En este caso, se usaron 5 de los 6 círculos para obtener un diámetro promedio de 218 px. Este valor se divide entre el diámetro real promedio en centímetros para obtener la densidad de píxeles por cm, que se usará para convertir píxeles a centímetros en análisis posteriores.

In [ ]:
cranberry.setup_measurements(detect_label = True,
                            diameter_cm = 1.7, 
                            plot = True)

A continuación, generamos una máscara binaria de frutos y lóculos con `generate_fruit_mask()`, donde los lóculos aparecerán en negro y el resto del fruto en blanco.

In [ ]:
cranberry.generate_fruit_mask()

Después, detectamos los frutos en la máscara con `detect_fruits()` y hacemos una inspección rápida con `plot=True`. La imagen muestra para cada fruto su contorno en verde, los contornos de sus lóculos en rosa y el área del pericarpio interno en cyan. Esta visualización permite corroborar la detección y determinar si es necesario ajustar los parámetros de segmentación.

In [ ]:
cranberry.detect_fruits(plot = True,
                       plot_size = (10,10))

Ahora, llevamos a cabo el analisis morfológico con `analyze_morphology`, el cual genera una copia de la imagen original con anotaciones para cada fruto y un DataFrame con los resultados. En ambos archivos, cada fruto es asignado a un identificador único (`id`) el cual es útil para comparar visualizar los datos numéricos.

In [ ]:
cranberry.analyze_morphology()

Alternativamente podemos explorar a detalle la segmentación de los diferentes tejidos de un fruto con `generate_single_fruit_masks`. Mediante el parámetro `fruit_id` podemos indicar exactamente cual fruto de la imagen o la tabla queremos visualizar. 

In [ ]:
cranberry.generate_single_fruit_masks(fruit_id = 2)

Finalmente, analizaremos el color de cada fruto. Por default, `analyze_color()` extrae información de color en los canales `rgb`, `lab`, `hsv` y `gray` para los tejidos `total_pericarp`, `outer_pericarp`, `internal_pericarp` y `locules`.

En este caso, excluiremos los lóculos del análisis, ya que al estar huecos, solo capturan el color negro del fondo. Para seleccionar tejidos específicos utilizamos el parámetro `tissue`, y para seleccionar canales de color específicos utilizamos el parámetro `color_space`.

Al pasar múltiples valores, estos deben escribirse separados por comas, en minúsculas y con los espacios reemplazados por `_`. Por ejemplo:

- Canales RGB y HSV: `rgb, hsv`
- Tejidos lóculos, pericarpio total y pericarpio externo: `locules, total_pericarp, outer_pericarp`

Si necesitamos mayor control sobre qué tejidos analizar, las máscaras obtenidas con `generate_single_fruit_masks()` nos permiten seleccionar únicamente los tejidos más relevantes para el análisis.

In [ ]:
tissues_ext = 'total_pericarp, outer_pericarp, internal_pericarp'

cranberry.analyze_color(tissue = tissues_ext,
                       color_space = 'rgb')

Al ejecutar `analyze_morphology()` y/o `analyze_color()`, se genera el objeto `results`, que contiene los resultados del análisis junto con los métodos necesarios para exportarlos. Para guardar todos los resultados en un solo paso, utilizamos `save_all()` como se muestra a continuación.

In [ ]:
cranberry.results.save_all()

## Alternativas:
#cranberry.results.save_img() # Guarda solo la imagen anotada
# cranberry.results.save_csv() # Guarda solo el o los CSV generados


También podemos exportar los parámetros utilizados en la sesión con `save_parameters()`, que genera dos archivos: uno `.txt` y uno `.json`. El archivo `.txt` está pensado para el usuario y contiene los parámetros de cada método, la versión de Traitly, la fecha y hora del análisis, y el nombre de la imagen. El archivo `.json`, en cambio, es útil para replicar el análisis, por ejemplo al procesar múltiples imágenes con `analyze_folder()` o al usar Traitly desde la terminal.

In [ ]:
cranberry.save_parameters()